# 📈 仮想通貨 次の足 上昇/下落 予測ツール
上から順に `Shift + Enter` で実行するだけ！

## ① ライブラリのインストール（初回のみ）

In [ ]:
!pip install requests pandas numpy matplotlib scikit-learn -q
print('インストール完了')

## ② インポート

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['axes.edgecolor']   = '#30363d'
plt.rcParams['axes.labelcolor']  = '#c9d1d9'
plt.rcParams['xtick.color']      = '#8b949e'
plt.rcParams['ytick.color']      = '#8b949e'
plt.rcParams['text.color']       = '#c9d1d9'
plt.rcParams['grid.color']       = '#21262d'
plt.rcParams['grid.linewidth']   = 0.5
plt.rcParams['legend.facecolor'] = '#1c2128'
plt.rcParams['legend.edgecolor'] = '#30363d'
print('インポート完了')

## ③ 設定 ← ここだけ変更すればOK

In [ ]:
COIN_ID = 'bitcoin'       # ビットコイン
# COIN_ID = 'ethereum'    # イーサリアム
# COIN_ID = 'solana'      # ソラナ
# COIN_ID = 'binancecoin' # BNB
# COIN_ID = 'ripple'      # XRP

DAYS = 60

print('設定完了 → 通貨:', COIN_ID, '/ 学習期間:', DAYS, '日')

## ④ データ取得

In [ ]:
def fetch_ohlc(coin_id, days):
    url = 'https://api.coingecko.com/api/v3/coins/' + coin_id + '/ohlc'
    r = requests.get(url, params={'vs_currency': 'usd', 'days': days}, timeout=15)
    r.raise_for_status()
    df = pd.DataFrame(r.json(), columns=['ts', 'open', 'high', 'low', 'close'])
    df['ts'] = pd.to_datetime(df['ts'], unit='ms')
    df.set_index('ts', inplace=True)
    return df.sort_index()

def fetch_current(coin_id):
    r = requests.get(
        'https://api.coingecko.com/api/v3/simple/price',
        params={'ids': coin_id, 'vs_currencies': 'usd',
                'include_24hr_change': 'true', 'include_market_cap': 'true'},
        timeout=10
    )
    r.raise_for_status()
    return r.json().get(coin_id, {})

print('データ取得中...')
df      = fetch_ohlc(COIN_ID, DAYS)
current = fetch_current(COIN_ID)
price   = current.get('usd', 0)
chg24   = current.get('usd_24h_change', 0)
mcap    = current.get('usd_market_cap', 0)
arrow   = 'UP' if chg24 >= 0 else 'DN'
sep     = '=' * 45
print(sep)
print('  通貨    :', COIN_ID.upper(), '/ USD')
print('  現在価格: $' + '{:,.2f}'.format(price))
print('  24H変動 :', arrow, '{:.2f}%'.format(abs(chg24)))
print('  時価総額: $' + '{:.1f}B'.format(mcap / 1e9))
print('  取得件数:', len(df), '本')
print('  期間    :', df.index[0].strftime('%Y/%m/%d'), '〜', df.index[-1].strftime('%Y/%m/%d'))
print(sep)

## ⑤ テクニカル指標の計算 & 機械学習

In [ ]:
def build_features(df):
    d = df.copy()
    for w in [3, 5, 10, 20]:
        d['ma' + str(w)] = d['close'].rolling(w).mean()
    for w in [1, 3, 5]:
        d['mom' + str(w)] = d['close'].pct_change(w)
    d['vol5']  = d['close'].pct_change().rolling(5).std()
    d['vol10'] = d['close'].pct_change().rolling(10).std()
    delta = d['close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['rsi']      = 100 - 100 / (1 + gain / (loss + 1e-9))
    mid = d['close'].rolling(20).mean()
    std = d['close'].rolling(20).std()
    d['bb_pos']       = (d['close'] - mid) / (2 * std + 1e-9)
    ema12             = d['close'].ewm(span=12, adjust=False).mean()
    ema26             = d['close'].ewm(span=26, adjust=False).mean()
    d['macd']         = ema12 - ema26
    d['macd_signal']  = d['macd'].ewm(span=9, adjust=False).mean()
    d['macd_diff']    = d['macd'] - d['macd_signal']
    d['hl_ratio']     = (d['high'] - d['low']) / (d['close'] + 1e-9)
    d['target']       = (d['close'].shift(-1) > d['close']).astype(int)
    return d

FEATURES = ['ma3','ma5','ma10','ma20','mom1','mom3','mom5',
            'vol5','vol10','rsi','bb_pos','macd','macd_signal','macd_diff','hl_ratio']

feat_df = build_features(df)
data    = feat_df.dropna()
X = data[FEATURES].values
y = data['target'].values

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_sc, y, test_size=0.2, shuffle=False)

clf = RandomForestClassifier(n_estimators=300, max_depth=6,
                             min_samples_leaf=3, random_state=42, n_jobs=-1)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
acc    = accuracy_score(y_te, y_pred)

print('学習完了')
print('バックテスト精度:', round(acc * 100, 1), '%')
print(classification_report(y_te, y_pred, target_names=['下落', '上昇']))

## ⑥ 予測結果

In [ ]:
latest_sc  = scaler.transform(data[FEATURES].iloc[[-1]].values)
pred       = clf.predict(latest_sc)[0]
prob       = clf.predict_proba(latest_sc)[0]
prob_up    = prob[1]
prob_dn    = prob[0]
confidence = max(prob_up, prob_dn) * 100

sep = '=' * 50
print(sep)
print('  予測結果  [' + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ']')
print(sep)
print('  現在価格     : $' + '{:,.2f}'.format(price))
print('  次の足の予測 :', '上昇 UP' if pred == 1 else '下落 DOWN')
print('  上昇確率     :', round(prob_up * 100, 1), '%')
print('  下落確率     :', round(prob_dn * 100, 1), '%')
print('  モデル信頼度 :', round(confidence, 1), '%')
print('  バックテスト精度:', round(acc * 100, 1), '%')
bar_len = 30
up_bar  = int(prob_up * bar_len)
dn_bar  = bar_len - up_bar
print()
print('  上昇 [' + 'X' * up_bar + '-' * dn_bar + ']', round(prob_up * 100, 1), '%')
print('  下落 [' + 'X' * dn_bar + '-' * up_bar + ']', round(prob_dn * 100, 1), '%')
print(sep)
print('  ※ このシグナルは参考情報です。投資判断は自己責任でお願いします。')

## ⑦ チャート表示

In [ ]:
GREEN  = '#3fb950'
RED    = '#f85149'
BLUE   = '#58a6ff'
AMBER  = '#e3b341'
PURPLE = '#bc8cff'
GRAY   = '#8b949e'

pred_color = GREEN if pred == 1 else RED
pred_label = 'UP' if pred == 1 else 'DOWN'

fig = plt.figure(figsize=(15, 11))
fig.suptitle(COIN_ID.upper() + '/USD $' + '{:,.2f}'.format(price)
             + '  ' + pred_label + '  信頼度' + str(round(confidence,1)) + '%',
             fontsize=13, fontweight='bold', color=pred_color, y=0.98)

gs  = GridSpec(4, 2, figure=fig, height_ratios=[3,1.2,1.2,1.2], hspace=0.08, wspace=0.25)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :], sharex=ax1)
ax3 = fig.add_subplot(gs[2, :], sharex=ax1)
ax4 = fig.add_subplot(gs[3, 0])
ax5 = fig.add_subplot(gs[3, 1])

ax1.plot(feat_df.index, feat_df['close'], color=BLUE, linewidth=1.3, label='終値', zorder=5)
ax1.plot(feat_df.index, feat_df['ma5'],   color=GREEN, linewidth=0.9, label='MA5')
ax1.plot(feat_df.index, feat_df['ma20'],  color=AMBER, linewidth=0.9, label='MA20')
bb_mid = feat_df['close'].rolling(20).mean()
bb_std = feat_df['close'].rolling(20).std()
ax1.plot(feat_df.index, bb_mid+2*bb_std, color=GRAY, linewidth=0.7, linestyle='--', alpha=0.5, label='BB')
ax1.plot(feat_df.index, bb_mid-2*bb_std, color=GRAY, linewidth=0.7, linestyle='--', alpha=0.5)
ax1.fill_between(feat_df.index, bb_mid+2*bb_std, bb_mid-2*bb_std, color=GRAY, alpha=0.06)
ax1.scatter(feat_df.index[-1], feat_df['close'].iloc[-1], color=pred_color, s=80, zorder=10)
ax1.set_ylabel('Price (USD)', fontsize=10)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '$' + '{:,.0f}'.format(x)))
ax1.legend(loc='upper left', fontsize=8, framealpha=0.4, ncol=6)
ax1.grid(True)

ax2.plot(feat_df.index, feat_df['rsi'], color=BLUE, linewidth=1.0)
ax2.axhline(70, color=RED,   linewidth=0.8, linestyle='--', alpha=0.7)
ax2.axhline(30, color=GREEN, linewidth=0.8, linestyle='--', alpha=0.7)
ax2.fill_between(feat_df.index, feat_df['rsi'], 70, where=(feat_df['rsi']>70), color=RED,   alpha=0.12)
ax2.fill_between(feat_df.index, feat_df['rsi'], 30, where=(feat_df['rsi']<30), color=GREEN, alpha=0.12)
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI (14)', fontsize=9)
ax2.grid(True)

ax3.plot(feat_df.index, feat_df['macd'],        color=BLUE,  linewidth=1.0, label='MACD')
ax3.plot(feat_df.index, feat_df['macd_signal'], color=AMBER, linewidth=1.0, label='Signal')
hist_c = [GREEN if v >= 0 else RED for v in feat_df['macd_diff']]
ax3.bar(feat_df.index, feat_df['macd_diff'], color=hist_c, alpha=0.55, width=0.03)
ax3.axhline(0, color=GRAY, linewidth=0.5)
ax3.set_ylabel('MACD', fontsize=9)
ax3.legend(loc='upper left', fontsize=7, framealpha=0.3, ncol=3)
ax3.grid(True)

for ax in [ax1, ax2]:
    plt.setp(ax.get_xticklabels(), visible=False)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax3.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.setp(ax3.get_xticklabels(), rotation=30, ha='right', fontsize=8)

ax4.pie([prob_up, prob_dn], labels=['上昇','下落'], colors=[GREEN, RED],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize':11},
        wedgeprops={'linewidth':1,'edgecolor':'#0d1117'})
ax4.set_title('予測確率', fontsize=11, pad=10)

importance = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
bar_c = [BLUE if v > importance.median() else GRAY for v in importance.values]
ax5.barh(importance.index, importance.values, color=bar_c, height=0.7)
ax5.set_title('特徴量の重要度', fontsize=11)
ax5.set_xlabel('Importance', fontsize=9)
ax5.grid(True, axis='x')

plt.savefig('crypto_prediction.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('チャートを crypto_prediction.png に保存しました')

## ⑧ 再予測（このセルだけ繰り返し実行）

In [ ]:
print('最新データを取得中...')
df      = fetch_ohlc(COIN_ID, DAYS)
current = fetch_current(COIN_ID)
price   = current.get('usd', 0)
chg24   = current.get('usd_24h_change', 0)
arrow   = 'UP' if chg24 >= 0 else 'DN'

feat_df   = build_features(df)
data      = feat_df.dropna()
latest_sc = scaler.transform(data[FEATURES].iloc[[-1]].values)
pred      = clf.predict(latest_sc)[0]
prob      = clf.predict_proba(latest_sc)[0]
prob_up   = prob[1]
prob_dn   = prob[0]
confidence = max(prob_up, prob_dn) * 100

sep = '=' * 50
print(sep)
print('  最新予測  [' + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ']')
print(sep)
print('  現在価格     : $' + '{:,.2f}'.format(price))
print('  次の足の予測 :', '上昇 UP' if pred == 1 else '下落 DOWN')
print('  上昇確率     :', round(prob_up * 100, 1), '%')
print('  下落確率     :', round(prob_dn * 100, 1), '%')
print('  モデル信頼度 :', round(confidence, 1), '%')
bar_len = 30
up_bar  = int(prob_up * bar_len)
dn_bar  = bar_len - up_bar
print()
print('  上昇 [' + 'X' * up_bar + '-' * dn_bar + ']', round(prob_up * 100, 1), '%')
print('  下落 [' + 'X' * dn_bar + '-' * up_bar + ']', round(prob_dn * 100, 1), '%')
print(sep)
print('  ※ このシグナルは参考情報です。投資判断は自己責任でお願いします。')